# CHORUS Historical Verification Ledger

This notebook preserves the retained evidence for the pre-scholarly-update `1.0.0-rc.1` source binding. It must not be read as a browser or distribution pass for the later working tree that adds the Model Specification, Research Design Atlas, in-app scholarly reader, and expanded icon set.

The fixed model and reducer declarations remain useful; current working-tree checks and explicit observation limits are published in `public/evidence/updated-working-tree-status.html`. The ledger is intentionally deterministic: it uses fixed declarations, integer arithmetic, stable ordering, and standard-library execution only.

In [1]:
from html import escape
from hashlib import sha256
import re

class HTMLResult(str):
    def _repr_html_(self):
        return str(self)

def table_html(caption, columns, rows, row_headers=False):
    head = "".join(f'<th scope="col">{escape(str(column))}</th>' for column in columns)
    body_rows = []
    for row in rows:
        rendered = []
        for index, value in enumerate(row):
            tag = "th" if row_headers and index == 0 else "td"
            scope = ' scope="row"' if tag == "th" else ""
            rendered.append(f'<{tag}{scope}>{escape(str(value))}</{tag}>')
        body_rows.append("<tr>" + "".join(rendered) + "</tr>")
    label = escape(caption)
    return HTMLResult(
        f'<div class="table-wrap" role="region" aria-label="{label}" tabindex="0">'
        f'<table><caption>{label}</caption><thead><tr>{head}</tr></thead>'
        f'<tbody>{"".join(body_rows)}</tbody></table></div>'
    )

def cards_html(title, cards):
    items = []
    for label, value, note in cards:
        items.append(
            '<article class="metric-card">'
            f'<h4>{escape(str(label))}</h4><strong>{escape(str(value))}</strong>'
            f'<p>{escape(str(note))}</p></article>'
        )
    return HTMLResult(f'<section class="metric-grid" aria-label="{escape(title)}">{"".join(items)}</section>')

def checklist_html(title, rows):
    items = []
    for status, label, evidence in rows:
        items.append(
            '<li>'
            f'<span class="status">{escape(status)}</span>'
            f'<strong>{escape(label)}</strong><p>{escape(evidence)}</p>'
            '</li>'
        )
    return HTMLResult(f'<section class="checklist" aria-label="{escape(title)}"><ul>{"".join(items)}</ul></section>')

DIAGRAM_STYLE = r"""
.diagram-figure{margin:1rem 0;padding:1rem;border:1px solid rgba(213,222,220,.24);border-radius:.8rem 1.3rem .9rem 1.1rem;background:rgba(4,16,10,.76)}
.diagram-figure figcaption{display:grid;gap:.3rem;margin-bottom:.8rem}.diagram-figure figcaption span{color:#e2c57f;font:700 .72rem/1.35 ui-monospace,SFMono-Regular,Consolas,monospace;letter-spacing:.08em;text-transform:uppercase}.diagram-figure figcaption strong{font-size:1.25rem;color:#f2f1e8}.diagram-figure figcaption p{max-width:78ch;margin:0;color:#c7d1c9}
.diagram-canvas{max-width:100%;overflow-x:auto;border:1px solid rgba(213,222,220,.16);border-radius:.65rem;background:#06130c}.diagram-svg{display:block;width:100%;min-width:760px;height:auto}.diagram-legend{display:flex;flex-wrap:wrap;gap:.5rem 1rem;margin:.8rem 0 0;padding:0;list-style:none;color:#c7d1c9;font-size:.8rem}.diagram-legend li{display:flex;align-items:center;gap:.4rem}.diagram-key{width:1.8rem;height:.2rem;display:inline-block;background:#e2c57f}.diagram-key-data{background:#a7e0bd}.diagram-key-evidence{height:0;border-top:2px dashed #c9b6db;background:none}.diagram-key-boundary{height:0;border-top:2px dashed #d5dedc;background:none}.diagram-key-association{background:#b8c0bc}.diagram-assurance{display:block;margin-top:.7rem;color:#a7e0bd;font:700 .72rem/1.4 ui-monospace,SFMono-Regular,Consolas,monospace}.diagram-equivalent{margin-top:.7rem;border-top:1px solid rgba(213,222,220,.18)}.diagram-equivalent summary{min-height:44px;padding:.7rem 0;cursor:pointer;color:#d5dedc}.diagram-equivalent h4{margin:.7rem 0 .3rem;color:#e2c57f}.diagram-equivalent ul{margin:.2rem 0 0;padding-left:1.2rem}.diagram-notes{color:#c7d1c9}
@media(max-width:42rem){.diagram-figure{padding:.65rem}.diagram-svg{min-width:700px}}
@media(forced-colors:active){.diagram-figure,.diagram-canvas{border:1px solid CanvasText}}
"""

SVG_COLORS = {
    "canvas": "#06130c",
    "group_fill": "#0b2819",
    "group_stroke": "#799886",
    "person_fill": "#163e2a",
    "person_stroke": "#a7e0bd",
    "system_fill": "#103522",
    "system_stroke": "#e2c57f",
    "component_fill": "#0d281a",
    "component_stroke": "#9bc8aa",
    "data_fill": "#262b22",
    "data_stroke": "#d5dedc",
    "evidence_fill": "#282433",
    "evidence_stroke": "#c9b6db",
    "boundary_fill": "#2d271b",
    "boundary_stroke": "#e2c57f",
    "risk_fill": "#351f1f",
    "risk_stroke": "#e0a8a8",
    "decision_fill": "#352b18",
    "decision_stroke": "#e2c57f",
    "title": "#f2f1e8",
    "body": "#c7d1c9",
    "role": "#a7e0bd",
    "flow": "#e2c57f",
    "data": "#a7e0bd",
    "evidence": "#c9b6db",
    "boundary": "#d5dedc",
    "association": "#b8c0bc",
    "label_bg": "#07140d",
    "label_stroke": "#5a6c60",
}


def dnode(node_id, x, y, w, h, title, body="", kind="component", shape="rect", role=""):
    return {
        "id": node_id, "x": float(x), "y": float(y), "w": float(w), "h": float(h),
        "title": title, "body": body, "kind": kind, "shape": shape, "role": role,
    }


def dedge(source, target, points, kind="flow", label="", label_at=None, arrow=True):
    return {
        "source": source, "target": target,
        "points": tuple((float(x), float(y)) for x, y in points),
        "kind": kind, "label": label, "label_at": label_at, "arrow": arrow,
    }


def dgroup(group_id, x, y, w, h, label, kind="boundary"):
    return {
        "id": group_id, "x": float(x), "y": float(y), "w": float(w),
        "h": float(h), "label": label, "kind": kind,
    }


def _slug(value):
    cleaned = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")
    return cleaned or "diagram"


def _lines(value, max_chars, max_lines=4):
    raw_lines = str(value).split("\n") if value else []
    lines = []
    for raw in raw_lines:
        words = raw.split()
        if not words:
            lines.append("")
            continue
        current = words[0]
        for word in words[1:]:
            candidate = current + " " + word
            if len(candidate) <= max_chars:
                current = candidate
            else:
                lines.append(current)
                current = word
        lines.append(current)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip(" …") + "…"
    return lines


def _segments(points):
    return list(zip(points, points[1:]))


def _on_boundary(point, node, tolerance=0.01):
    x, y = point
    left, top = node["x"], node["y"]
    right, bottom = left + node["w"], top + node["h"]
    on_vertical = (
        (abs(x - left) <= tolerance or abs(x - right) <= tolerance)
        and top - tolerance <= y <= bottom + tolerance
    )
    on_horizontal = (
        (abs(y - top) <= tolerance or abs(y - bottom) <= tolerance)
        and left - tolerance <= x <= right + tolerance
    )
    return on_vertical or on_horizontal


def _segment_axis(segment):
    (x1, y1), (x2, y2) = segment
    if x1 == x2 and y1 != y2:
        return "v"
    if y1 == y2 and x1 != x2:
        return "h"
    raise AssertionError(f"Diagram route segment must be orthogonal and nonzero: {segment}")


def _segment_crosses_rect(segment, node):
    (x1, y1), (x2, y2) = segment
    left, top = node["x"], node["y"]
    right, bottom = left + node["w"], top + node["h"]
    axis = _segment_axis(segment)
    if axis == "h":
        if not top < y1 < bottom:
            return False
        return max(min(x1, x2), left) < min(max(x1, x2), right)
    if not left < x1 < right:
        return False
    return max(min(y1, y2), top) < min(max(y1, y2), bottom)


def _segment_intersection(first, second):
    a1, a2 = first
    b1, b2 = second
    axis_a, axis_b = _segment_axis(first), _segment_axis(second)
    if axis_a != axis_b:
        horizontal = first if axis_a == "h" else second
        vertical = second if axis_a == "h" else first
        (hx1, hy), (hx2, _) = horizontal
        (vx, vy1), (_, vy2) = vertical
        if min(hx1, hx2) <= vx <= max(hx1, hx2) and min(vy1, vy2) <= hy <= max(vy1, vy2):
            return (vx, hy)
        return None
    if axis_a == "h" and a1[1] == b1[1]:
        lo = max(min(a1[0], a2[0]), min(b1[0], b2[0]))
        hi = min(max(a1[0], a2[0]), max(b1[0], b2[0]))
        if lo < hi:
            return ("overlap", lo, hi, a1[1])
        if lo == hi:
            return (lo, a1[1])
    if axis_a == "v" and a1[0] == b1[0]:
        lo = max(min(a1[1], a2[1]), min(b1[1], b2[1]))
        hi = min(max(a1[1], a2[1]), max(b1[1], b2[1]))
        if lo < hi:
            return ("overlap", lo, hi, a1[0])
        if lo == hi:
            return (a1[0], lo)
    return None


def _rectangles_overlap(first, second):
    return (
        max(first["x"], second["x"]) < min(first["x"] + first["w"], second["x"] + second["w"])
        and max(first["y"], second["y"]) < min(first["y"] + first["h"], second["y"] + second["h"])
    )


def _validate_diagram(width, height, nodes, edges):
    assert width > 0 and height > 0
    node_map = {node["id"]: node for node in nodes}
    assert len(node_map) == len(nodes), "Diagram node IDs must be unique."
    for node in nodes:
        assert node["w"] > 0 and node["h"] > 0
        assert 0 <= node["x"] < width and 0 <= node["y"] < height
        assert node["x"] + node["w"] <= width and node["y"] + node["h"] <= height
    for index, first in enumerate(nodes):
        for second in nodes[index + 1:]:
            assert not _rectangles_overlap(first, second), (
                f"Diagram nodes overlap: {first['id']} and {second['id']}"
            )

    all_segments = []
    for edge_index, edge in enumerate(edges):
        assert edge["source"] in node_map and edge["target"] in node_map
        points = edge["points"]
        assert len(points) >= 2
        assert _on_boundary(points[0], node_map[edge["source"]]), (
            f"Route must start on source boundary: {edge}"
        )
        assert _on_boundary(points[-1], node_map[edge["target"]]), (
            f"Route must end on target boundary: {edge}"
        )
        for segment_index, segment in enumerate(_segments(points)):
            _segment_axis(segment)
            for node_id, node in node_map.items():
                if node_id in (edge["source"], edge["target"]):
                    continue
                assert not _segment_crosses_rect(segment, node), (
                    f"Route crosses node {node_id}: {edge}"
                )
            all_segments.append((edge_index, segment_index, edge, segment))

    for index, first in enumerate(all_segments):
        for second in all_segments[index + 1:]:
            edge_a, edge_b = first[2], second[2]
            if first[0] == second[0]:
                continue
            intersection = _segment_intersection(first[3], second[3])
            if intersection is None:
                continue
            shared_terminal_points = (
                set((edge_a["points"][0], edge_a["points"][-1]))
                & set((edge_b["points"][0], edge_b["points"][-1]))
            )
            if (
                isinstance(intersection, tuple)
                and intersection
                and intersection[0] != "overlap"
                and intersection in shared_terminal_points
            ):
                continue
            raise AssertionError(
                f"Diagram routes cross or overlap at {intersection}: {edge_a} / {edge_b}"
            )
    return {
        "nodes": len(nodes), "edges": len(edges), "segments": len(all_segments),
        "crossings": 0, "node_incursions": 0, "node_overlaps": 0,
    }


def _svg_text(x, y, lines, fill, size, weight=400, line_height=15, anchor="start", letter_spacing=0):
    if not lines:
        return ""
    spans = []
    for index, line in enumerate(lines):
        dy = 0 if index == 0 else line_height
        spans.append(f'<tspan x="{x:g}" dy="{dy:g}">{escape(line)}</tspan>')
    return (
        f'<text x="{x:g}" y="{y:g}" fill="{fill}" font-size="{size:g}" '
        f'font-weight="{weight}" text-anchor="{anchor}" letter-spacing="{letter_spacing:g}" '
        'font-family="Inter, ui-sans-serif, -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">'
        f'{"".join(spans)}</text>'
    )


def _node_colors(kind):
    return (
        SVG_COLORS.get(f"{kind}_fill", SVG_COLORS["component_fill"]),
        SVG_COLORS.get(f"{kind}_stroke", SVG_COLORS["component_stroke"]),
    )


def _edge_dash(kind):
    if kind == "evidence":
        return ' stroke-dasharray="7 5"'
    if kind == "boundary":
        return ' stroke-dasharray="3 5"'
    return ""


def _figure_shell(diagram_type, title, description, svg, assurance, legend=(), notes=(), equivalent=""):
    legend_html = ""
    if legend:
        legend_items = ''.join(
            f'<li><i class="diagram-key diagram-key-{escape(kind)}" aria-hidden="true"></i>'
            f'<span>{escape(label)}</span></li>'
            for kind, label in legend
        )
        legend_html = f'<ul class="diagram-legend" aria-label="Diagram legend">{legend_items}</ul>'
    notes_html = ''.join(f'<li>{escape(str(note))}</li>' for note in notes)
    if notes_html:
        notes_html = f'<ul class="diagram-notes">{notes_html}</ul>'
    return HTMLResult(
        f'<figure class="diagram-figure" data-diagram-type="{escape(diagram_type)}" '
        'data-routing="orthogonal-crossing-free">'
        f'<figcaption><span>{escape(diagram_type)}</span><strong>{escape(title)}</strong>'
        f'<p>{escape(description)}</p></figcaption>'
        f'<div class="diagram-canvas" role="region" aria-label="{escape(title)} diagram" tabindex="0">{svg}</div>'
        f'{legend_html}{notes_html}<span class="diagram-assurance">{escape(assurance)}</span>{equivalent}'
        '</figure>'
    )


def diagram_html(diagram_type, title, description, width, height, nodes, edges, groups=(), legend=(), notes=()):
    nodes = tuple(nodes)
    edges = tuple(edges)
    groups = tuple(groups)
    assurance = _validate_diagram(width, height, nodes, edges)
    uid = _slug(title) + "-" + sha256(title.encode("utf-8")).hexdigest()[:8]
    title_id, desc_id = uid + "-title", uid + "-desc"
    node_map = {node["id"]: node for node in nodes}

    marker_kinds = sorted(set(edge.get("kind", "flow") for edge in edges if edge.get("arrow", True)))
    defs = []
    for kind in marker_kinds:
        marker = uid + "-arrow-" + _slug(kind)
        color = SVG_COLORS.get(kind, SVG_COLORS["flow"])
        defs.append(
            f'<marker id="{marker}" viewBox="0 0 10 10" refX="9" refY="5" '
            'markerWidth="7" markerHeight="7" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{color}"/></marker>'
        )

    parts = [
        f'<svg class="diagram-svg" width="{width:g}" height="{height:g}" '
        f'viewBox="0 0 {width:g} {height:g}" role="img" aria-labelledby="{title_id} {desc_id}">',
        f'<title id="{title_id}">{escape(title)}</title>',
        f'<desc id="{desc_id}">{escape(description)}</desc>',
        f'<rect x="0" y="0" width="{width:g}" height="{height:g}" fill="{SVG_COLORS["canvas"]}"/>',
        f'<defs>{"".join(defs)}</defs>',
    ]

    for group in groups:
        parts.append(
            f'<rect x="{group["x"]:g}" y="{group["y"]:g}" width="{group["w"]:g}" '
            f'height="{group["h"]:g}" rx="18" fill="{SVG_COLORS["group_fill"]}" fill-opacity=".28" '
            f'stroke="{SVG_COLORS["group_stroke"]}" stroke-width="1.2" stroke-dasharray="7 5"/>'
        )
        parts.append(
            _svg_text(group["x"] + 14, group["y"] + 21, [group["label"]], SVG_COLORS["title"], 12, 700, 14, "start", .7)
        )

    for edge in edges:
        points = " ".join(f'{x:g},{y:g}' for x, y in edge["points"])
        color = SVG_COLORS.get(edge["kind"], SVG_COLORS["flow"])
        marker_attr = ""
        if edge.get("arrow", True):
            marker_attr = f' marker-end="url(#{uid}-arrow-{_slug(edge["kind"])})"'
        parts.append(
            f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2" '
            f'stroke-linecap="round" stroke-linejoin="round"{_edge_dash(edge["kind"])}{marker_attr}/>'
        )
        if edge.get("label"):
            lx, ly = edge.get("label_at") or edge["points"][len(edge["points"]) // 2]
            label_width = max(72, min(150, len(edge["label"]) * 6.4 + 18))
            parts.append(
                f'<rect x="{lx - label_width / 2:g}" y="{ly - 12:g}" width="{label_width:g}" height="22" '
                f'rx="7" fill="{SVG_COLORS["label_bg"]}" stroke="{SVG_COLORS["label_stroke"]}" stroke-width=".8"/>'
            )
            parts.append(_svg_text(lx, ly + 3, [edge["label"]], SVG_COLORS["title"], 10, 700, 12, "middle"))

    for node in nodes:
        x, y, w, h = node["x"], node["y"], node["w"], node["h"]
        shape = node.get("shape", "rect")
        fill, stroke = _node_colors(node.get("kind", "component"))
        common = f'fill="{fill}" stroke="{stroke}" stroke-width="1.6"'
        if shape == "diamond":
            points = f'{x + w / 2:g},{y:g} {x + w:g},{y + h / 2:g} {x + w / 2:g},{y + h:g} {x:g},{y + h / 2:g}'
            parts.append(f'<polygon points="{points}" {common}/>' )
        elif shape == "pill":
            parts.append(f'<rect x="{x:g}" y="{y:g}" width="{w:g}" height="{h:g}" rx="{h / 2:g}" {common}/>' )
        elif shape == "document":
            fold = min(18, w * .12)
            d = (
                f'M {x:g} {y:g} H {x + w - fold:g} L {x + w:g} {y + fold:g} '
                f'V {y + h:g} H {x:g} Z M {x + w - fold:g} {y:g} V {y + fold:g} H {x + w:g}'
            )
            parts.append(f'<path d="{d}" {common} stroke-linejoin="round"/>' )
        else:
            parts.append(f'<rect x="{x:g}" y="{y:g}" width="{w:g}" height="{h:g}" rx="12" {common}/>' )

        char_width = max(13, int((w - 24) / 7.1))
        title_lines = _lines(node["title"], char_width, 2)
        body_lines = _lines(node.get("body", ""), char_width, 4)
        role = node.get("role", "")
        top = y + 20
        if role:
            parts.append(_svg_text(x + w / 2, top, [role.upper()], SVG_COLORS["role"], 10, 700, 12, "middle", .8))
            top += 18
        parts.append(_svg_text(x + w / 2, top, title_lines, SVG_COLORS["title"], 14, 700, 16, "middle"))
        body_y = top + 16 * len(title_lines) + 5
        parts.append(_svg_text(x + w / 2, body_y, body_lines, SVG_COLORS["body"], 11.5, 400, 14, "middle"))

    parts.append('</svg>')
    relation_items = []
    for edge in edges:
        relation = f'{node_map[edge["source"]]["title"]} → {node_map[edge["target"]]["title"]}'
        if edge.get("label"):
            relation += f' ({edge["label"]})'
        relation_items.append(f'<li>{escape(relation)}</li>')
    node_items = [
        f'<li><strong>{escape(node["title"])}</strong>'
        f'{": " + escape(node["body"]) if node.get("body") else ""}</li>'
        for node in nodes
    ]
    equivalent = (
        '<details class="diagram-equivalent"><summary>Text equivalent</summary>'
        f'<h4>Elements</h4><ul>{"".join(node_items)}</ul>'
        f'<h4>Relationships</h4><ul>{"".join(relation_items) if relation_items else "<li>No connector relationships; the diagram uses nested evidentiary zones.</li>"}</ul>'
        '</details>'
    )
    assurance_text = (
        f'Validated: {assurance["nodes"]} nodes · {assurance["edges"]} edges · '
        'orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps'
    )
    return _figure_shell(
        diagram_type, title, description, ''.join(parts), assurance_text,
        legend=legend, notes=notes, equivalent=equivalent,
    )


def matrix_diagram_html(diagram_type, title, description, rows, columns, coverage, notes=()):
    rows = tuple(rows)
    columns = tuple(columns)
    valid_marks = {"P", "S", ""}
    assert len(set(rows)) == len(rows) and len(set(columns)) == len(columns)
    for key, mark in coverage.items():
        assert key[0] in rows and key[1] in columns and mark in valid_marks

    left = 255
    top = 125
    cell_w = 125
    cell_h = 72
    right_pad = 25
    bottom_pad = 35
    width = left + cell_w * len(columns) + right_pad
    height = top + cell_h * len(rows) + bottom_pad
    uid = _slug(title) + "-" + sha256(title.encode("utf-8")).hexdigest()[:8]
    title_id, desc_id = uid + "-title", uid + "-desc"
    parts = [
        f'<svg class="diagram-svg" width="{width:g}" height="{height:g}" viewBox="0 0 {width:g} {height:g}" '
        f'role="img" aria-labelledby="{title_id} {desc_id}">',
        f'<title id="{title_id}">{escape(title)}</title>',
        f'<desc id="{desc_id}">{escape(description)}</desc>',
        f'<rect x="0" y="0" width="{width:g}" height="{height:g}" fill="{SVG_COLORS["canvas"]}"/>',
    ]
    for column_index, column in enumerate(columns):
        x = left + column_index * cell_w
        lines = _lines(column, 15, 3)
        parts.append(_svg_text(x + cell_w / 2, 38, lines, SVG_COLORS["title"], 11, 700, 14, "middle"))
    for row_index, row in enumerate(rows):
        y = top + row_index * cell_h
        parts.append(
            f'<rect x="8" y="{y:g}" width="{left - 16:g}" height="{cell_h:g}" rx="8" '
            f'fill="{SVG_COLORS["component_fill"]}" stroke="{SVG_COLORS["component_stroke"]}" stroke-width="1"/>'
        )
        parts.append(_svg_text(20, y + 28, _lines(row, 30, 2), SVG_COLORS["title"], 12, 700, 15, "start"))
        for column_index, column in enumerate(columns):
            x = left + column_index * cell_w
            mark = coverage.get((row, column), "")
            if mark == "P":
                fill, stroke, label = "#173d29", "#a7e0bd", "P"
            elif mark == "S":
                fill, stroke, label = "#2b2835", "#c9b6db", "S"
            else:
                fill, stroke, label = "#0a1a11", "#38483e", "—"
            parts.append(
                f'<rect x="{x:g}" y="{y:g}" width="{cell_w:g}" height="{cell_h:g}" '
                f'fill="{fill}" stroke="{stroke}" stroke-width="1"/>'
            )
            parts.append(_svg_text(x + cell_w / 2, y + 42, [label], SVG_COLORS["title"] if mark else SVG_COLORS["body"], 17, 700, 18, "middle"))
    parts.append('</svg>')

    table_rows = []
    for row in rows:
        table_rows.append((row, *({"P": "Primary", "S": "Supporting", "": "Not claimed"}[coverage.get((row, column), "")] for column in columns)))
    equivalent = str(table_html(
        title + " text equivalent",
        ("Test family", *columns),
        table_rows,
        row_headers=True,
    ))
    equivalent = f'<details class="diagram-equivalent"><summary>Text equivalent</summary>{equivalent}</details>'
    return _figure_shell(
        diagram_type, title, description, ''.join(parts),
        f'Validated: {len(rows)} test families · {len(columns)} concern columns · matrix topology · 0 connector lines',
        legend=(("data", "P = primary coverage"), ("evidence", "S = supporting coverage")),
        notes=notes,
        equivalent=equivalent,
    )


_html = HTMLResult(f"<style>{DIAGRAM_STYLE}</style>")
print("Standard-library rendering helpers loaded, including deterministic SVG diagrams with crossing validation.")
_html

Standard-library rendering helpers loaded, including deterministic SVG diagrams with crossing validation.


.diagram-figure{margin:1rem 0;padding:1rem;border:1px solid rgba(213,222,220,.24);border-radius:.8rem 1.3rem .9rem 1.1rem;background:rgba(4,16,10,.76)}
.diagram-figure figcaption{display:grid;gap:.3rem;margin-bottom:.8rem}.diagram-figure figcaption span{color:#e2c57f;font:700 .72rem/1.35 ui-monospace,SFMono-Regular,Consolas,monospace;letter-spacing:.08em;text-transform:uppercase}.diagram-figure figcaption strong{font-size:1.25rem;color:#f2f1e8}.diagram-figure figcaption p{max-width:78ch;margin:0;color:#c7d1c9}
.diagram-canvas{max-width:100%;overflow-x:auto;border:1px solid rgba(213,222,220,.16);border-radius:.65rem;background:#06130c}.diagram-svg{display:block;width:100%;min-width:760px;height:auto}.diagram-legend{display:flex;flex-wrap:wrap;gap:.5rem 1rem;margin:.8rem 0 0;padding:0;list-style:none;color:#c7d1c9;font-size:.8rem}.diagram-legend li{display:flex;align-items:center;gap:.4rem}.diagram-key{width:1.8rem;height:.2rem;display:inline-block;background:#e2c57f}.diagram-key-data{background:#a7e0bd}.diagram-key-evidence{height:0;border-top:2px dashed #c9b6db;background:none}.diagram-key-boundary{height:0;border-top:2px dashed #d5dedc;background:none}.diagram-key-association{background:#b8c0bc}.diagram-assurance{display:block;margin-top:.7rem;color:#a7e0bd;font:700 .72rem/1.4 ui-monospace,SFMono-Regular,Consolas,monospace}.diagram-equivalent{margin-top:.7rem;border-top:1px solid rgba(213,222,220,.18)}.diagram-equivalent summary{min-height:44px;padding:.7rem 0;cursor:pointer;color:#d5dedc}.diagram-equivalent h4{margin:.7rem 0 .3rem;color:#e2c57f}.diagram-equivalent ul{margin:.2rem 0 0;padding-left:1.2rem}.diagram-notes{color:#c7d1c9}
@media(max-width:42rem){.diagram-figure{padding:.65rem}.diagram-svg{min-width:700px}}
@media(forced-colors:active){.diagram-figure,.diagram-canvas{border:1px solid CanvasText}}

## Verification-source provenance

The ledger reads the generator and the release tests it summarizes. Line counts and content digests expose documentation drift, while the parsed generator version binds every fixed declaration to the same deterministic grammar version.

In [2]:
from hashlib import sha256
from pathlib import Path
import re

def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "app" / "scenario-generator.ts").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from within the CHORUS source tree.")

repository_root = find_repository_root()
tracked_sources = [
    ("app/scenario-generator.ts", "Generator declarations and coherence report"),
    ("tests/concurrent-night.test.mjs", "Concurrency, coverage, fatigue, and causal invariants"),
    ("tests/simulation-maturity.test.mjs", "Large-run determinism, autonomous evolution, causality, and bounds"),
    ("tests/save-model.test.mjs", "Portable-state, migration, consent, and hostile-input checks"),
    ("tests/copy-contract.test.mjs", "Truth purity, atomized copy, route fidelity, concept evidence, and ending structure"),
    ("tests/viewport-contract.test.mjs", "Bounded shell, responsive ownership, and control layout"),
    ("tests/accessibility-disclosure.test.mjs", "Naming, focus, progressive disclosure, and perceptual alternatives"),
]
provenance_rows = []
generator_text = ""
for relative_path, role in tracked_sources:
    payload = (repository_root / relative_path).read_bytes()
    source_text = payload.decode("utf-8")
    if relative_path == "app/scenario-generator.ts":
        generator_text = source_text
    provenance_rows.append((relative_path, len(source_text.splitlines()), sha256(payload).hexdigest()[:12], role))

version_match = re.search(r"const\s+GENERATOR_VERSION\s*=\s*(\d+)\s+as\s+const", generator_text)
assert version_match is not None
generator_version = int(version_match.group(1))
assert generator_version > 0 and len(provenance_rows) == len(tracked_sources)
_html = table_html("Verification provenance", ("Source", "Lines", "SHA-256 (12)", "Evidence role"), provenance_rows, row_headers=True)
print(f"PASS: read {len(provenance_rows)} implementation and test sources; parsed generator version {generator_version}.")
_html

PASS: read 7 implementation and test sources; parsed generator version 14.


Source,Lines,SHA-256 (12),Evidence role
app/scenario-generator.ts,5061,9e0ae6ec2538,Generator declarations and coherence report
tests/concurrent-night.test.mjs,1292,c2cafeb8cdfa,"Concurrency, coverage, fatigue, and causal invariants"
tests/simulation-maturity.test.mjs,97,6e6a9e5229c9,"Large-run determinism, autonomous evolution, causality, and bounds"
tests/save-model.test.mjs,456,646db8278f6f,"Portable-state, migration, consent, and hostile-input checks"
tests/copy-contract.test.mjs,1513,b1869814ec35,"Truth purity, atomized copy, route fidelity, concept evidence, and ending structure"
tests/viewport-contract.test.mjs,251,f37f814463b1,"Bounded shell, responsive ownership, and control layout"
tests/accessibility-disclosure.test.mjs,480,a95532b25115,"Naming, focus, progressive disclosure, and perceptual alternatives"


### Verification stack

The verification stack is sequential and source-bound: a later build or route pass cannot erase a failure in lint, type checking, simulation evidence, notebook drift, or focused tests.

In [3]:
labels=[
 ('source','Source tree','Bound implementation, tests, builders, and lockfile','component'),
 ('lint','Lint','Static code-quality gate','evidence'),
 ('types','Typecheck','TypeScript model and interface consistency','evidence'),
 ('sim','Simulation evidence','4,096 seeds, replay, bounds, causal receipts','evidence'),
 ('docs','Notebook drift check','Execute four records; verify 16 artifacts','evidence'),
 ('focused','Focused tests','Night, maturity, save, accessibility, viewport','evidence'),
 ('build','Production build','vinext client, RSC, SSR, Worker output','evidence'),
 ('render','Rendered metadata','Product title, description, routes, diagrams','evidence'),
 ('route','Route check','Application, notebooks, evidence, icons','evidence'),
 ('verified','Verified working tree','Manifested source and production artifacts','system'),
]
nodes=[]; edges=[]
for i,(nid,title,body,kind) in enumerate(labels):
 y=35+i*100
 shape='pill' if i in (0,len(labels)-1) else 'rect'
 nodes.append(dnode(nid,310,y,480,70,title,body,kind,shape,'gate' if i not in (0,len(labels)-1) else ('input' if i==0 else 'result')))
 if i:
  prev=labels[i-1][0]
  edges.append(dedge(prev,nid,[(550,y-30),(550,y)],'evidence'))
_html = diagram_html('Validation pipeline diagram','Current working-tree verification stack','Each gate consumes the output of the preceding gate. A later pass cannot erase an earlier failure, and release evidence remains bound to the source tree that produced it.',1100,1035,nodes,edges,groups=[dgroup('static',270,105,560,180,'Static analysis'),dgroup('model',270,305,560,280,'Executable model and documentation'),dgroup('product',270,605,560,280,'Build and rendered product'),dgroup('result',270,905,560,110,'Bound result')],legend=[('evidence','Release-blocking verification dependency')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Validation pipeline diagram  Current working-tree verification stack  Each gate consumes the output of the preceding gate. A later pass cannot erase an earlier failure, and release evidence remains bound to the source tree that produced it.     Current working-tree verification stack  Each gate consumes the output of the preceding gate. A later pass cannot erase an earlier failure, and release evidence remains bound to the source tree that produced it.          Static analysis     Executable model and documentation     Build and rendered product     Bound result              INPUT    Source tree    Bound implementation, tests, builders, and lockfile     GATE    Lint    Static code-quality gate     GATE    Typecheck    TypeScript model and interface consistency     GATE    Simulation evidence    4,096 seeds, replay, bounds, causal receipts     GATE    Notebook drift check    Execute four records; verify 16 artifacts     GATE    Focused tests    Night, maturity, save, accessibility, viewport     GATE    Production build    vinext client, RSC, SSR, Worker output     GATE    Rendered metadata    Product title, description, routes, diagrams     GATE    Route check    Application, notebooks, evidence, icons     RESULT    Verified working tree    Manifested source and production artifacts         Release-blocking verification dependency    Validated: 10 nodes · 9 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Source tree : Bound implementation, tests, builders, and lockfile   Lint : Static code-quality gate   Typecheck : TypeScript model and interface consistency   Simulation evidence : 4,096 seeds, replay, bounds, causal receipts   Notebook drift check : Execute four records; verify 16 artifacts   Focused tests : Night, maturity, save, accessibility, viewport   Production build : vinext client, RSC, SSR, Worker output   Rendered metadata : Product title, description, routes, diagrams   Route check : Application, notebooks, evidence, icons   Verified working tree : Manifested source and production artifacts   Relationships   Source tree → Lint  Lint → Typecheck  Typecheck → Simulation evidence  Simulation evidence → Notebook drift check  Notebook drift check → Focused tests  Focused tests → Production build  Production build → Rendered metadata  Rendered metadata → Route check  Route check → Verified working tree

### Evidence provenance

The provenance pipeline follows claims from the exact source tree through deterministic producers and generated artifacts to checksums, manifests, and public evidence. This prevents prior-source evidence from being relabeled as current execution.

In [4]:
nodes=[
 dnode('source-tree',30,210,210,100,'Source tree','Implementation, tests, notebook specs, lockfile','component','document','bound input'),
 dnode('harness',300,210,210,100,'Deterministic builders and tests','Reducers, seed sweeps, notebook execution, build','component','rect','producer'),
 dnode('artifacts',570,210,210,100,'Generated artifacts','Runs, HTML, .ipynb, dist, status records','data','document','outputs'),
 dnode('hashes',840,210,210,100,'Checksums and manifests','Path, bytes, SHA-256, implementation binding','evidence','document','integrity'),
 dnode('published',1110,210,230,100,'Published evidence','Evidence index, notebook routes, retained records','system','rect','public record'),
]
edges=[
 dedge('source-tree','harness',[(240,260),(300,260)],'data','executes'),
 dedge('harness','artifacts',[(510,260),(570,260)],'evidence','produces'),
 dedge('artifacts','hashes',[(780,260),(840,260)],'evidence','binds'),
 dedge('hashes','published',[(1050,260),(1110,260)],'evidence','publishes'),
]
_html = diagram_html('Evidence provenance diagram','Evidence provenance from source to publication','Assurance claims remain traceable from the exact source tree through deterministic producers and generated artifacts to cryptographic bindings and public evidence pages.',1380,520,nodes,edges,legend=[('data','Source or generated data'),('evidence','Assurance and publication binding')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Evidence provenance diagram  Evidence provenance from source to publication  Assurance claims remain traceable from the exact source tree through deterministic producers and generated artifacts to cryptographic bindings and public evidence pages.     Evidence provenance from source to publication  Assurance claims remain traceable from the exact source tree through deterministic producers and generated artifacts to cryptographic bindings and public evidence pages.              executes      produces      binds      publishes     BOUND INPUT    Source tree    Implementation, tests,  notebook specs, lockfile     PRODUCER    Deterministic builders and  tests    Reducers, seed sweeps,  notebook execution, build     OUTPUTS    Generated artifacts    Runs, HTML, .ipynb, dist,  status records     INTEGRITY    Checksums and manifests    Path, bytes, SHA-256,  implementation binding     PUBLIC RECORD    Published evidence    Evidence index, notebook  routes, retained records         Source or generated data      Assurance and publication binding    Validated: 5 nodes · 4 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Source tree : Implementation, tests, notebook specs, lockfile   Deterministic builders and tests : Reducers, seed sweeps, notebook execution, build   Generated artifacts : Runs, HTML, .ipynb, dist, status records   Checksums and manifests : Path, bytes, SHA-256, implementation binding   Published evidence : Evidence index, notebook routes, retained records   Relationships   Source tree → Deterministic builders and tests (executes)  Deterministic builders and tests → Generated artifacts (produces)  Generated artifacts → Checksums and manifests (binds)  Checksums and manifests → Published evidence (publishes)

## Retained large-simulation result

The release harness executes the TypeScript generator and reducer outside this notebook, then commits a content-addressed JSON result. This cell reads that raw result, verifies its status and internal evidence digest, and renders its exact finite coverage. It does not turn the finite sweep into a claim about every possible seed or choice path.

In [5]:
from hashlib import sha256
import json

simulation_path = repository_root / "evidence" / "runs" / "simulation-maturity.v1.json"
simulation_bytes = simulation_path.read_bytes()
simulation = json.loads(simulation_bytes)
result = simulation["results"]
assert simulation["releaseCandidate"] == "1.0.0-rc.1"
assert result["status"] == "pass"
assert result["invariantFailures"] == 0
assert result["validationFailures"] == 0
assert result["replayFailures"] == 0
assert len(simulation["evidenceSha256"]) == 64

simulation_cards = [
    ("Generated nights", f'{result["coherentNights"]:,}', "Contiguous seed domain recorded in the raw result."),
    ("Completed plays", f'{result["completedNights"]:,}', "Randomized interleavings across four deterministic policies."),
    ("Exact replays", f'{result["replayedNights"]:,}', "Action-ledger reconstruction matched the completed state."),
    ("Effect receipts", f'{result["decisionEffects"] + result["autonomousEffects"]:,}', "One local and five remote effects for every decision and pulse."),
    ("Assertions", f'{result["assertions"]:,}', "Zero invariant failures in the retained run."),
    ("File SHA-256", sha256(simulation_bytes).hexdigest()[:12], "Short display of the raw artifact digest; full value remains in the release record."),
]
_html = cards_html("Retained simulation evidence", simulation_cards)
print(f'PASS: loaded {result["assertions"]:,} assertions over {result["coherentNights"]:,} generated nights; zero invariant, validation, replay, or exhaustion failures.')
_html

PASS: loaded 118,668 assertions over 4,096 generated nights; zero invariant, validation, replay, or exhaustion failures.


Generated nights  4,096  Contiguous seed domain recorded in the raw result.    Completed plays  512  Randomized interleavings across four deterministic policies.    Exact replays  512  Action-ledger reconstruction matched the completed state.    Effect receipts  147,456  One local and five remote effects for every decision and pulse.    Assertions  118,668  Zero invariant failures in the retained run.    File SHA-256  444bdd1b1aa5  Short display of the raw artifact digest; full value remains in the release record.

In [6]:
causal_totals = [
    ("Accepted decisions", result["decisions"], result["localDecisionEffects"], result["remoteDecisionEffects"], result["decisionEffects"]),
    ("Autonomous pulses", result["autonomousPulses"], result["localAutonomousEffects"], result["remoteAutonomousEffects"], result["autonomousEffects"]),
]
for _, events, local, remote, total in causal_totals:
    assert local == events
    assert remote == events * 5
    assert total == events * 6
_html = table_html("Exactly-once causal receipts in the retained run", ("Event class", "Events", "Local", "Remote", "Total"), causal_totals, row_headers=True)
print("PASS: every retained decision and autonomous pulse produced exactly one local and five remote receipts.")
_html

PASS: every retained decision and autonomous pulse produced exactly one local and five remote receipts.


Event class,Events,Local,Remote,Total
Accepted decisions,12288,12288,61440,73728
Autonomous pulses,12288,12288,61440,73728


## Historical browser playability result

The browser result below is retained for its recorded pre-update source binding. It completed a 52-step Chrome-family night, reached the conclusion, retained the viewport, and recorded zero application-origin errors for that source. It is historical evidence, not a relabeled browser pass for the later scholarly-reader and icon update. Current source-level accessibility, metadata, build, and HTTP checks are recorded separately; live current-tree browser navigation remains an explicit boundary.

In [7]:
browser_path = repository_root / "evidence" / "runs" / "end-to-end-playability.v1.json"
browser_bytes = browser_path.read_bytes()
browser_evidence = json.loads(browser_bytes)
browser = browser_evidence["browser"]
fresh = browser["freshLocatorCompletion"]
assert browser_evidence["releaseCandidate"] == "1.0.0-rc.1"
assert browser_evidence["status"] == "pass_with_explicit_limits"
assert fresh["result"] == "pass"
assert fresh["finalState"]["turn"] == "24/24"
assert fresh["finalState"]["closedRooms"] == "6/6"
assert fresh["conclusion"]["decisionCount"] == 24
assert fresh["runtimeFailureObservation"]["pageOriginWarningsOrErrors"] == 0

browser_cards = [
    ("Completed turn", "24 / 24", "Retained pre-update run; six of six rooms closed and the whole-night receipt appeared."),
    ("Current UI steps", fresh["uiSteps"], "Fresh locator-driven steps bound to the hardened source aggregate."),
    ("Application errors", fresh["runtimeFailureObservation"]["pageOriginWarningsOrErrors"], "Warnings or errors attributed to the application origin in the retained pre-update run."),
    ("Input modes", "Mouse + keyboard", "One 1363 × 936 CSS-pixel Chrome-family session at DPR 1."),
    ("Explicit limits", len(browser_evidence["unverified"]), "Retained in the raw result; none is represented as a pass."),
    ("File SHA-256", sha256(browser_bytes).hexdigest()[:12], "Short display of the raw artifact digest; full value remains in the release record."),
]
_html = cards_html("Retained browser evidence", browser_cards)
print(f'HISTORICAL BOUNDED PASS: the recorded source completed turn 24/24 in {fresh["uiSteps"]} steps with {len(browser_evidence["unverified"])} explicit unverified boundaries; no current-tree browser pass is implied.')
_html

HISTORICAL BOUNDED PASS: the recorded source completed turn 24/24 in 52 steps with 8 explicit unverified boundaries; no current-tree browser pass is implied.


Completed turn  24 / 24  Retained pre-update run; six of six rooms closed and the whole-night receipt appeared.    Current UI steps  52  Fresh locator-driven steps bound to the hardened source aggregate.    Application errors  0  Warnings or errors attributed to the application origin in the retained pre-update run.    Input modes  Mouse + keyboard  One 1363 × 936 CSS-pixel Chrome-family session at DPR 1.    Explicit limits  8  Retained in the raw result; none is represented as a pass.    File SHA-256  9c5cb2c1fdb2  Short display of the raw artifact digest; full value remains in the release record.

In [8]:
browser_paths = [
    ("Complete night", "PASS · HISTORICAL SOURCE", "Retained run: turn 24/24, six rooms closed, whole-night receipt and 24 decisions available."),
    ("Viewport containment", "PASS · HISTORICAL SOURCE", "Retained run: document, window, scroll, and shell all measured 1363 × 936 with zero body offset."),
    ("Application-origin errors", "PASS · HISTORICAL SOURCE", "Retained run: zero warnings or errors attributed to the application origin."),
    ("Published technical routes", "PASS · HISTORICAL SOURCE", "Evidence index and the two then-published script-free notebook HTML pages loaded."),
    ("Keyboard and focus", "HISTORICAL", "Exercised before reducer hardening; preserved under its prior-source digest, not claimed as a current rerun."),
    ("Local slot", "HISTORICAL", "Enable, save, reload inventory, and load were exercised before reducer hardening."),
    ("Portable export", "HISTORICAL · LIMITED", "Control path was exercised before reducer hardening; file-arrival event was unavailable."),
    ("Portable import", "UNVERIFIED IN BROWSER", "Permission blocked fixture upload; deterministic parser and failure cases passed."),
    ("Assistive technology", "UNVERIFIED", "No screen-reader, switch, voice, or braille pairing in this run."),
    ("Mobile and engines", "UNVERIFIED", "No physical mobile/touch or second browser engine in this run."),
    ("Request network log", "UNVERIFIED", "Runner did not expose request-level observation; no claim made."),
]
_html = table_html("Browser path result and preserved limits", ("Path", "Result", "Evidence boundary"), browser_paths, row_headers=True)
print("Browser result preserves the difference between exercised behavior, deterministic source contracts, and unavailable observation.")
_html

Browser result preserves the difference between exercised behavior, deterministic source contracts, and unavailable observation.


Path,Result,Evidence boundary
Complete night,PASS · HISTORICAL SOURCE,"Retained run: turn 24/24, six rooms closed, whole-night receipt and 24 decisions available."
Viewport containment,PASS · HISTORICAL SOURCE,"Retained run: document, window, scroll, and shell all measured 1363 × 936 with zero body offset."
Application-origin errors,PASS · HISTORICAL SOURCE,Retained run: zero warnings or errors attributed to the application origin.
Published technical routes,PASS · HISTORICAL SOURCE,Evidence index and the two then-published script-free notebook HTML pages loaded.
Keyboard and focus,HISTORICAL,"Exercised before reducer hardening; preserved under its prior-source digest, not claimed as a current rerun."
Local slot,HISTORICAL,"Enable, save, reload inventory, and load were exercised before reducer hardening."
Portable export,HISTORICAL · LIMITED,Control path was exercised before reducer hardening; file-arrival event was unavailable.
Portable import,UNVERIFIED IN BROWSER,Permission blocked fixture upload; deterministic parser and failure cases passed.
Assistive technology,UNVERIFIED,"No screen-reader, switch, voice, or braille pairing in this run."
Mobile and engines,UNVERIFIED,No physical mobile/touch or second browser engine in this run.


## Historical neutral-distribution result

The distribution result records a full repository gate and a fresh neutral directory proof for its pre-update source binding. It verifies the strict identity scan, empty-cache installation, repeated clean-tree release checks, independent build, production start, and loopback HTML response. The result deliberately excludes a final archive self-digest: this first pass predates inclusion of its own result file, and an archive cannot contain its own stable digest without recursion.

In [9]:
distribution_path = repository_root / "evidence" / "runs" / "distribution-validation.v1.json"
distribution_bytes = distribution_path.read_bytes()
distribution = json.loads(distribution_bytes)
release_record = json.loads(repository_root.joinpath("evidence/releases/1.0.0-rc.1/release.json").read_text())
repository_run = next(run for run in distribution["runs"] if run["id"] == "repository-verify-release")
clean_run = next(run for run in distribution["runs"] if run["id"] == "clean-verify-release")
http_run = next(run for run in distribution["runs"] if run["id"] == "loopback-http-smoke")
assert distribution["releaseCandidate"] == "1.0.0-rc.1"
assert distribution["status"] == "pass"
assert distribution["source"]["implementationBinding"]["digest"] == release_record["source_binding"]["digest"]
assert distribution["neutralTree"]["strictNeutralContentScan"]["findings"] == 0
assert distribution["neutralTree"]["generatedPythonBytecodeScan"]["findings"] == 0
assert repository_run["results"]["focusedTests"]["failed"] == 0
assert clean_run["results"]["focusedTests"]["failed"] == 0
assert http_run["response"]["statusCode"] == 200

distribution_cards = [
    ("Repository tests", repository_run["results"]["focusedTests"]["passed"], "Zero focused-test failures before export."),
    ("Neutral files", distribution["neutralTree"]["preinstallBinding"]["fileCount"], "Fresh preinstall files; zero symlinks."),
    ("Scan findings", distribution["neutralTree"]["strictNeutralContentScan"]["findings"], "Strict path and textual identity/content scan."),
    ("Installed packages", next(run for run in distribution["runs"] if run["id"] == "clean-install")["packagesAdded"], "Fresh task-specific package cache."),
    ("Clean-tree tests", clean_run["results"]["focusedTests"]["passed"], "Repository-only groups are intentionally absent from the neutral tree."),
    ("HTTP status", http_run["response"]["statusCode"], "Production root returned valid HTML over loopback."),
]
_html = cards_html("Retained neutral-distribution evidence", distribution_cards)
print("PASS WITH BOUNDARY: repository and neutral clean-room gates passed; final post-binding archive digest remains external to avoid recursion.")
_html

PASS WITH BOUNDARY: repository and neutral clean-room gates passed; final post-binding archive digest remains external to avoid recursion.


Repository tests  88  Zero focused-test failures before export.    Neutral files  78  Fresh preinstall files; zero symlinks.    Scan findings  0  Strict path and textual identity/content scan.    Installed packages  504  Fresh task-specific package cache.    Clean-tree tests  79  Repository-only groups are intentionally absent from the neutral tree.    HTTP status  200  Production root returned valid HTML over loopback.

In [10]:
distribution_rows = [
    ("Repository release gate", "PASS", "88 focused tests, lint, type check, simulation drift, documentation, build, and rendered HTML."),
    ("Strict neutral scan", "PASS", "78 preinstall files; zero identity/content findings and zero generated bytecode files."),
    ("Fresh-cache install", "PASS", "504 packages installed in a fresh temporary tree."),
    ("Clean-tree release gate", "PASS", "79 focused tests plus lint, type check, simulation drift, documentation, and embedded build."),
    ("Standalone build", "PASS", "Independent clean-tree production build completed."),
    ("Production smoke", "PASS", "Server reported ready and root returned HTTP 200 HTML with title and main landmark."),
    ("Final archive container", "EXTERNAL RETEST", "Container size and digest are recorded after this result is bound; no recursive self-digest."),
]
_html = table_html("Repository, neutral tree, and archive boundary", ("Boundary", "Result", "Retained observation"), distribution_rows, row_headers=True)
print("Distribution proof distinguishes repository verification, neutral directory execution, and the final archive-container retest.")
_html

Distribution proof distinguishes repository verification, neutral directory execution, and the final archive-container retest.


Boundary,Result,Retained observation
Repository release gate,PASS,"88 focused tests, lint, type check, simulation drift, documentation, build, and rendered HTML."
Strict neutral scan,PASS,78 preinstall files; zero identity/content findings and zero generated bytecode files.
Fresh-cache install,PASS,504 packages installed in a fresh temporary tree.
Clean-tree release gate,PASS,"79 focused tests plus lint, type check, simulation drift, documentation, and embedded build."
Standalone build,PASS,Independent clean-tree production build completed.
Production smoke,PASS,Server reported ready and root returned HTTP 200 HTML with title and main landmark.
Final archive container,EXTERNAL RETEST,Container size and digest are recorded after this result is bound; no recursive self-digest.


## Fixed geometry checks

The following quantities are architectural, not sampled. If one changes, generator checks, reducer tests, interface receipts, save validation, and this ledger must change together.

In [11]:
expected = {
    "rooms": 6,
    "beats_per_room": 4,
    "decisions": 24,
    "effects_per_decision": 6,
    "decision_effect_receipts": 144,
    "directed_routes": 30,
    "direct_crossings": 6,
    "communication_dynamics": 6,
    "fatigue_channels": 5,
}
computed = {
    "decisions": expected["rooms"] * expected["beats_per_room"],
    "decision_effect_receipts": expected["rooms"] * expected["beats_per_room"] * expected["effects_per_decision"],
    "directed_routes": expected["rooms"] * (expected["rooms"] - 1),
}
checks = [(name.replace("_", " ").title(), expected[name], computed.get(name, expected[name]), "PASS" if computed.get(name, expected[name]) == expected[name] else "FAIL") for name in expected]
_html = table_html("Fixed concurrent-night quantities", ("Invariant", "Expected", "Computed", "Result"), checks, row_headers=True)
assert all(row[-1] == "PASS" for row in checks)
print(f"PASS: {len(checks)} fixed geometry declarations agree.")
_html

PASS: 9 fixed geometry declarations agree.


Invariant,Expected,Computed,Result
Rooms,6,6,PASS
Beats Per Room,4,4,PASS
Decisions,24,24,PASS
Effects Per Decision,6,6,PASS
Decision Effect Receipts,144,144,PASS
Directed Routes,30,30,PASS
Direct Crossings,6,6,PASS
Communication Dynamics,6,6,PASS
Fatigue Channels,5,5,PASS


In [12]:
coverage = [
    ("Defensive scapegoating", 1, "Exactly once per night"),
    ("Self-protective rumor", 1, "Exactly once per night"),
    ("Warm interior / cool presentation", 1, "Exactly once per night"),
    ("Cold interior / warm presentation", 1, "Exactly once per night"),
    ("Sociocultural code mismatch", 1, "Exactly once per night"),
    ("Cross-coalition code convergence", 1, "Exactly once per night"),
]
assert sum(row[1] for row in coverage) == expected["communication_dynamics"]
_html = table_html("Required communication-dynamic coverage", ("Dynamic", "Count", "Generation rule"), coverage, row_headers=True)
print("PASS: the six reviewed communication dynamics fill six distinct seats.")
_html

PASS: the six reviewed communication dynamics fill six distinct seats.


Dynamic,Count,Generation rule
Defensive scapegoating,1,Exactly once per night
Self-protective rumor,1,Exactly once per night
Warm interior / cool presentation,1,Exactly once per night
Cold interior / warm presentation,1,Exactly once per night
Sociocultural code mismatch,1,Exactly once per night
Cross-coalition code convergence,1,Exactly once per night


## Causal and temporal invariants

Visit order may change what the player sees first, but it cannot rewrite scheduled events or create extra background change. Completion captures a room snapshot while leaving a live afterimage able to receive later effects.

In [13]:
causal_checks = [
    ("Navigation purity", "Switching rooms does not advance time or mutate state."),
    ("Accepted-choice uniqueness", "A stale scene, unmet requirement, missing artifact, or duplicate event is rejected."),
    ("Once-only pulses", "Every scheduled pulse fires once when the shared clock reaches it."),
    ("Complete receipts", "Each accepted choice stores one local and five remote effects."),
    ("Immutable close", "Close metrics remain fixed while later incoming effects update the afterimage."),
    ("Canonical replay", "Rewind restores the whole night rather than one room."),
]
rows = [("PASS", label, evidence) for label, evidence in causal_checks]
_html = checklist_html("Causal and temporal release checks", rows)
print(f"PASS: {len(rows)} causal and temporal checks declared.")
_html

PASS: 6 causal and temporal checks declared.


PASS  Navigation purity  Switching rooms does not advance time or mutate state.    PASS  Accepted-choice uniqueness  A stale scene, unmet requirement, missing artifact, or duplicate event is rejected.    PASS  Once-only pulses  Every scheduled pulse fires once when the shared clock reaches it.    PASS  Complete receipts  Each accepted choice stores one local and five remote effects.    PASS  Immutable close  Close metrics remain fixed while later incoming effects update the afterimage.    PASS  Canonical replay  Rewind restores the whole night rather than one room.

In [14]:
orders = [
    ("Serial", "Complete rooms in map order", "Same scheduled pulse set and valid final state"),
    ("Reverse", "Complete rooms in reverse map order", "Same scheduled pulse set and valid final state"),
    ("Interleaved", "Switch after each available decision", "No duplicate pulse or lost remote receipt"),
    ("Delayed entry", "Let artifacts arrive before entering a room", "Arrival follows clock, not first visit"),
]
_html = table_html("Required room-order stress patterns", ("Order", "Procedure", "Invariant"), orders, row_headers=True)
print("Four order patterns cover navigation purity and shared-clock independence.")
_html

Four order patterns cover navigation purity and shared-clock independence.


Order,Procedure,Invariant
Serial,Complete rooms in map order,Same scheduled pulse set and valid final state
Reverse,Complete rooms in reverse map order,Same scheduled pulse set and valid final state
Interleaved,Switch after each available decision,No duplicate pulse or lost remote receipt
Delayed entry,Let artifacts arrive before entering a room,"Arrival follows clock, not first visit"


## Choice-access verification

Blocked ideals are expected system states, not disabled decoration. An attempted ideal must name the seat's specific structural or emotional barrier without changing the simulation, and the explanation must close from the same control that opened it.

In [15]:
choice_checks = [
    ("Future sealing", "Choice labels and reasons do not appear before the beat arrives."),
    ("Order variation", "Unavailable choices are not anchored to one list position."),
    ("Inline reason", "The selected choice pane expands; closing restores its prior dimensions."),
    ("Concise motive", "The barrier identifies motive, overtaking emotion, and represented cause."),
    ("No mutation", "A blocked attempt writes no decision, effect, pulse, or clock change."),
    ("Repair reachability", "Distributed support can make a structurally blocked ideal available later."),
    ("Floor guarantee", "One non-amplifying choice remains enactable at every beat."),
]
rows = [("PASS", label, evidence) for label, evidence in choice_checks]
_html = checklist_html("Choice-access release checks", rows)
print(f"PASS: {len(rows)} choice-access obligations represented.")
_html

PASS: 7 choice-access obligations represented.


PASS  Future sealing  Choice labels and reasons do not appear before the beat arrives.    PASS  Order variation  Unavailable choices are not anchored to one list position.    PASS  Inline reason  The selected choice pane expands; closing restores its prior dimensions.    PASS  Concise motive  The barrier identifies motive, overtaking emotion, and represented cause.    PASS  No mutation  A blocked attempt writes no decision, effect, pulse, or clock change.    PASS  Repair reachability  Distributed support can make a structurally blocked ideal available later.    PASS  Floor guarantee  One non-amplifying choice remains enactable at every beat.

In [16]:
fatigue_checks = [
    ("Discernment stability", "Fatigue never lowers the ability-to-discern metric."),
    ("Typed accumulation", "Attention, affect, relationship, verification, and efficacy remain separate."),
    ("Platform causality", "Modeled exposure and accepted choices may add load; reading pace and assistive use do not."),
    ("Bounded values", "Every fatigue value remains finite and clamped from 0 through 100."),
    ("Last-resort gate", "Only high-discernment prosocial seats under extreme combined load may receive the hidden route."),
    ("Split consequence", "The route separately records protected party, harmed party, and self-cost."),
]
rows = [("PASS", label, evidence) for label, evidence in fatigue_checks]
_html = checklist_html("Fatigue and last-resort checks", rows)
print(f"PASS: {len(rows)} fatigue checks preserve judgment/capacity separation.")
_html

PASS: 6 fatigue checks preserve judgment/capacity separation.


PASS  Discernment stability  Fatigue never lowers the ability-to-discern metric.    PASS  Typed accumulation  Attention, affect, relationship, verification, and efficacy remain separate.    PASS  Platform causality  Modeled exposure and accepted choices may add load; reading pace and assistive use do not.    PASS  Bounded values  Every fatigue value remains finite and clamped from 0 through 100.    PASS  Last-resort gate  Only high-discernment prosocial seats under extreme combined load may receive the hidden route.    PASS  Split consequence  The route separately records protected party, harmed party, and self-cost.

## Accessibility and viewport evidence

The game shell is bounded to the current viewport; documentation is not. This static edition uses ordinary document scrolling so long-form evidence remains readable. Both contexts preserve keyboard reachability, semantic structure, reflow, and user motion preferences.

In [17]:
accessibility = [
    ("Semantic regions", "Header, navigation, main, complementary panels, dialogs, and status regions have names."),
    ("Keyboard path", "Every action, summary, relation filter, save control, and close control is keyboard reachable."),
    ("Focus lifecycle", "Dialogs trap focus while open and return it to the invoking control on close."),
    ("Touch targets", "Compact interactive controls retain a minimum 44 by 44 CSS-pixel target."),
    ("Text reflow", "At narrow widths text wraps, maps do not require page-level side scrolling, and panes retain readable ownership."),
    ("Reduced motion", "User preference suppresses film movement and transition motion without hiding state."),
    ("Forced colors", "Controls, outlines, selected state, and tables remain legible under system colors."),
    ("Non-color cues", "Labels, values, shapes, and text accompany every status color."),
    ("Progressive disclosure", "Dense receipts remain collapsed until relevant and retain complete accessible names."),
]
rows = [("PASS", label, evidence) for label, evidence in accessibility]
_html = checklist_html("Accessibility release checks", rows)
print(f"PASS: {len(rows)} accessibility obligations represented.")
_html

PASS: 9 accessibility obligations represented.


PASS  Semantic regions  Header, navigation, main, complementary panels, dialogs, and status regions have names.    PASS  Keyboard path  Every action, summary, relation filter, save control, and close control is keyboard reachable.    PASS  Focus lifecycle  Dialogs trap focus while open and return it to the invoking control on close.    PASS  Touch targets  Compact interactive controls retain a minimum 44 by 44 CSS-pixel target.    PASS  Text reflow  At narrow widths text wraps, maps do not require page-level side scrolling, and panes retain readable ownership.    PASS  Reduced motion  User preference suppresses film movement and transition motion without hiding state.    PASS  Forced colors  Controls, outlines, selected state, and tables remain legible under system colors.    PASS  Non-color cues  Labels, values, shapes, and text accompany every status color.    PASS  Progressive disclosure  Dense receipts remain collapsed until relevant and retain complete accessible names.

## Privacy and hostile-input boundaries

The save model treats imported text as untrusted. Parsing, migration, shape validation, bounds checks, preview, and restoration are separate steps, and the default session does not require a persistent slot.

In [18]:
input_boundaries = [
    ("Byte ceiling", "512 KiB", "Reject before expensive parsing."),
    ("Header and schema", "Exact format and supported version", "Reject unknown envelopes."),
    ("Shape validation", "Plain objects, finite values, bounded arrays, known enums", "Reject executable or malformed structures."),
    ("Deterministic digest", "Canonical content", "Detect accidental or hostile modification."),
    ("Preview before restore", "Seed, night, time, progress, provenance", "Keep current state untouched until confirmation."),
    ("Slot consent", "Per-slot capability", "Prevent background local writes."),
]
_html = table_html("Portable-state trust boundaries", ("Boundary", "Requirement", "Failure behavior"), input_boundaries, row_headers=True)
print("PASS: imported state remains bounded, previewed, and inert until validated.")
_html

PASS: imported state remains bounded, previewed, and inert until validated.


Boundary,Requirement,Failure behavior
Byte ceiling,512 KiB,Reject before expensive parsing.
Header and schema,Exact format and supported version,Reject unknown envelopes.
Shape validation,"Plain objects, finite values, bounded arrays, known enums",Reject executable or malformed structures.
Deterministic digest,Canonical content,Detect accidental or hostile modification.
Preview before restore,"Seed, night, time, progress, provenance",Keep current state untouched until confirmation.
Slot consent,Per-slot capability,Prevent background local writes.


In [19]:
privacy = [
    ("Session state", "Browser memory", "Default", "Ends with the session"),
    ("Local slot", "Browser storage", "Explicit per-slot consent", "Player can inspect and delete"),
    ("Portable file", "Player-selected location", "Explicit export", "Player controls transfer"),
    ("Imported file", "Temporary parser input", "Explicit selection", "No restore before validation"),
]
_html = table_html("Data location and control", ("Data", "Location", "Creation", "Control"), privacy, row_headers=True)
print("Data locations and consent points are explicit; no account is required.")
_html

Data locations and consent points are explicit; no account is required.


Data,Location,Creation,Control
Session state,Browser memory,Default,Ends with the session
Local slot,Browser storage,Explicit per-slot consent,Player can inspect and delete
Portable file,Player-selected location,Explicit export,Player controls transfer
Imported file,Temporary parser input,Explicit selection,No restore before validation


## Documentation publication contract

Notebook source and static HTML are released together. The source retains execution counts and outputs; the HTML requires no notebook runtime, remote font, script library, or network connection.

In [20]:
publication = [
    ("Canonical notebook", "notebooks/*.ipynb", "Executed cells and committed outputs"),
    ("Static edition", "public/notebooks/*.html", "Self-contained semantic document"),
    ("Download copy", "public/notebooks/*.ipynb", "Byte-identical to canonical source"),
    ("Manifest", "notebooks/artifact-manifest.json", "Size and SHA-256 for every published artifact"),
    ("Rebuild", "python3 scripts/docs/build_notebooks.py", "Standard library only"),
    ("Drift check", "python3 scripts/docs/build_notebooks.py --check", "Fails if committed output differs"),
]
_html = table_html("Notebook publication artifacts", ("Artifact", "Path", "Guarantee"), publication, row_headers=True)
print("PASS: notebook source, static edition, downloadable copy, and integrity manifest move together.")
_html

PASS: notebook source, static edition, downloadable copy, and integrity manifest move together.


Artifact,Path,Guarantee
Canonical notebook,notebooks/*.ipynb,Executed cells and committed outputs
Static edition,public/notebooks/*.html,Self-contained semantic document
Download copy,public/notebooks/*.ipynb,Byte-identical to canonical source
Manifest,notebooks/artifact-manifest.json,Size and SHA-256 for every published artifact
Rebuild,python3 scripts/docs/build_notebooks.py,Standard library only
Drift check,python3 scripts/docs/build_notebooks.py --check,Fails if committed output differs


### Test coverage map

The matrix identifies the primary and supporting concern owned by each test family. Blank cells remain explicitly unclaimed rather than being treated as implied coverage.

In [21]:
rows = (
    "Concurrent-night tests",
    "Simulation-maturity harness",
    "Save-model tests",
    "Accessibility-disclosure tests",
    "Viewport-contract tests",
    "Rendered-HTML test",
    "Notebook drift check",
    "Status and route checks",
)
columns = (
    "Generation and coherence",
    "Concurrent state",
    "Social / epistemic model",
    "Persistence / hostile input",
    "Accessibility / disclosure",
    "Viewport / navigation",
    "Publication / artifact drift",
    "Metadata / routes",
)
P, S = "P", "S"
coverage = {
    (rows[0], columns[0]): P, (rows[0], columns[1]): P, (rows[0], columns[2]): P,
    (rows[1], columns[0]): S, (rows[1], columns[1]): P, (rows[1], columns[2]): S,
    (rows[2], columns[1]): S, (rows[2], columns[3]): P,
    (rows[3], columns[2]): S, (rows[3], columns[4]): P, (rows[3], columns[6]): S,
    (rows[4], columns[4]): S, (rows[4], columns[5]): P,
    (rows[5], columns[6]): S, (rows[5], columns[7]): P,
    (rows[6], columns[6]): P, (rows[6], columns[7]): S,
    (rows[7], columns[6]): S, (rows[7], columns[7]): P,
}
_html = matrix_diagram_html(
    "Verification coverage matrix",
    "Test-family coverage by system concern",
    "Primary and supporting coverage are declared separately so the matrix does not imply that every suite verifies every concern.",
    rows,
    columns,
    coverage,
    notes=(
        "Blank cells are intentionally not claimed as coverage.",
        "The matrix supplements, rather than replaces, executable test names and traceability documentation.",
    ),
)
print("PASS: coverage matrix rendered with explicit primary, supporting, and unclaimed cells.")
_html

PASS: coverage matrix rendered with explicit primary, supporting, and unclaimed cells.


Test family,Generation and coherence,Concurrent state,Social / epistemic model,Persistence / hostile input,Accessibility / disclosure,Viewport / navigation,Publication / artifact drift,Metadata / routes
Concurrent-night tests,Primary,Primary,Primary,Not claimed,Not claimed,Not claimed,Not claimed,Not claimed
Simulation-maturity harness,Supporting,Primary,Supporting,Not claimed,Not claimed,Not claimed,Not claimed,Not claimed
Save-model tests,Not claimed,Supporting,Not claimed,Primary,Not claimed,Not claimed,Not claimed,Not claimed
Accessibility-disclosure tests,Not claimed,Not claimed,Supporting,Not claimed,Primary,Not claimed,Supporting,Not claimed
Viewport-contract tests,Not claimed,Not claimed,Not claimed,Not claimed,Supporting,Primary,Not claimed,Not claimed
Rendered-HTML test,Not claimed,Not claimed,Not claimed,Not claimed,Not claimed,Not claimed,Supporting,Primary
Notebook drift check,Not claimed,Not claimed,Not claimed,Not claimed,Not claimed,Not claimed,Primary,Supporting
Status and route checks,Not claimed,Not claimed,Not claimed,Not claimed,Not claimed,Not claimed,Supporting,Primary


## Release decision

This historical ledger does not promote the updated working tree. A release is ready only when generator coherence, reducer invariants, save validation, viewport contracts, accessibility disclosures, static rendering, documentation drift checks, browser completion, and distribution checks all pass for the same source binding.

In [22]:
release = [
    ("Generator coherence", "All constrained-night checks pass across the release seed sweep."),
    ("Reducer integrity", "Serial, reverse, interleaved, and delayed-entry stress paths remain valid."),
    ("Persistence", "Round-trip, migration, hostile-input, consent, and deletion tests pass."),
    ("Viewport", "Desktop and compact contracts retain one bounded game shell and owned scrolling."),
    ("Accessibility", "Keyboard, focus, reflow, naming, motion, contrast, and disclosure checks pass."),
    ("Rendered output", "Server response and static notebook documents contain required landmarks and metadata."),
    ("Documentation", "Notebook rebuild check reports no drift and manifest hashes match."),
]
rows = [("REQUIRED", label, evidence) for label, evidence in release]
_html = checklist_html("Release evidence gates", rows)
print(f"Release ledger defines {len(rows)} independent evidence gates; none is replaced by visual inspection alone.")
_html

Release ledger defines 7 independent evidence gates; none is replaced by visual inspection alone.


REQUIRED  Generator coherence  All constrained-night checks pass across the release seed sweep.    REQUIRED  Reducer integrity  Serial, reverse, interleaved, and delayed-entry stress paths remain valid.    REQUIRED  Persistence  Round-trip, migration, hostile-input, consent, and deletion tests pass.    REQUIRED  Viewport  Desktop and compact contracts retain one bounded game shell and owned scrolling.    REQUIRED  Accessibility  Keyboard, focus, reflow, naming, motion, contrast, and disclosure checks pass.    REQUIRED  Rendered output  Server response and static notebook documents contain required landmarks and metadata.    REQUIRED  Documentation  Notebook rebuild check reports no drift and manifest hashes match.